## Extra calculations
**Aims:** Generating other supplementary statistics from ASF Installer Survey data<br>
**Author:** Elysia Lucas<br>
**Date:** 2024-05-14<br>

In [4]:
with open("data_setup.py") as file:
    exec(file.read())
"""
Running "data_setup.py" does the following:
- Imports survey question numbers lookups class as col
- Imports cleaned, anonymised survey analytical sample data as data
- Adds new columns to the data dataframe for each sub-population category,
column names are:
-- EmploymentType
-- CompanySizeOwnerV2
-- CompanySizeEmployeeV2
-- SectorTime
-- NumberInstalls
-- DesiredIncrease
- Defines the following functions to generate cross tables:
-- crosstable(subpop, dataframe, x, ans)
-- location_crosstab(sq_col)
-- transform(x, answer_list, column1, column2)
-- explode_select_all(column)
- Defines the following functions to generate figures:
-- stackedbar(df, number, question, section)
-- groupedbar(df, number, question, section)
-- donut(df, number, question, section)
"""

with open("free_text_recode.py") as file:
    exec(file.read())
"""
Running "free_text_recode.py" re-assigns 'Other' free text responses to Select all that apply type questions
by re-assigning them to a  new or existing answer, or keeping them as 'Other'
"""

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Columns: 331 entries, 0a. Response ID to 115. To what extent do you agree with this statement? "My work in the heat pump sector negatively affects my mental health."
dtypes: bool(1), category(199), datetime64[ns](2), float64(1), int16(1), object(127)
memory usage: 980.8+ KB


'\nRunning "data_setup.py" does the following:\n- Imports survey question numbers lookups class as col\n- Imports cleaned, anonymised survey analytical sample data as data\n- Adds new columns to the data dataframe for each sub-population category,\ncolumn names are:\n-- EmploymentType\n-- CompanySizeOwnerV2\n-- CompanySizeEmployeeV2\n-- SectorTime\n-- NumberInstalls\n-- DesiredIncrease\n- Defines the following functions to generate cross tables:\n-- crosstable(subpop, dataframe, x, ans)\n-- location_crosstab(sq_col)\n-- transform(x, answer_list, column1, column2)\n-- explode_select_all(column)\n- Defines the following functions to generate figures:\n-- stackedbar(df, number, question, section)\n-- groupedbar(df, number, question, section)\n-- donut(df, number, question, section)\n'

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Columns: 331 entries, 0a. Response ID to 115. To what extent do you agree with this statement? "My work in the heat pump sector negatively affects my mental health."
dtypes: bool(1), category(199), datetime64[ns](2), float64(1), int16(1), object(127)
memory usage: 980.8+ KB


'\nRunning "free_text_recode.py" re-assigns \'Other\' free text responses to Select all that apply type questions\nby re-assigning them to a  new or existing answer, or keeping them as \'Other\'\n'

In [34]:
from matplotlib import pyplot as plt
import statistics

### Estimating number of heat pumps installed per employee

In [70]:
# Company owners excl. sole traders
companysize_hpinstalls = data[[col.q6a, col.q37a, col.q37b]]
companysize_hpinstalls.columns = ['CompanySize','Q37a','Q37b']
companysize_hpinstalls = companysize_hpinstalls.drop(companysize_hpinstalls[companysize_hpinstalls.CompanySize == "Not asked"].index)
companysize_hpinstalls

# companysize_hpinstalls.to_csv("../../outputs/other-statistics/companysize_hpinstalls.csv")
# pd.crosstab(companysize_hpinstalls.iloc[:,0], companysize_hpinstalls.iloc[:,1]).to_csv("../../outputs/other-statistics/companysize_hpinstalls_1.csv")
# pd.crosstab(companysize_hpinstalls.iloc[:,0], companysize_hpinstalls.iloc[:,2]).to_csv("../../outputs/other-statistics/companysize_hpinstalls_2.csv")

,CompanySize,Q37a,Q37b
0,I own a company with 5 or fewer employees,9 or fewer,Not asked
3,I own a company with 5 or fewer employees,25 to 49,Not asked
4,I own a company with 26-100 employees,150 to 349,Not asked
5,I own a company with 5 or fewer employees,9 or fewer,Not asked
6,I own a company with 5 or fewer employees,10 to 24,Not asked
...,...,...,...
336,I own a company with 5 or fewer employees,10 to 24,Not asked
340,I own a company with 6-25 employees,50 to 99,Not asked
341,I own a company with 6-25 employees,25 to 49,Not asked
343,I own a company with 5 or fewer employees,25 to 49,Not asked


In [88]:
company_size_mapping = {"I’m a sole trader": 1,
                        'I own a company with 5 or fewer employees': statistics.median([1,5]),
                        'I own a company with 6-25 employees': statistics.median([6,25]),
                        'I own a company with 26-100 employees': statistics.median([26,100]),
                        'I own a company with over 100 employees': 100}

hp_number_mapping = {'None': 0,
                     '9 or fewer': statistics.median([1,9]),
                     '10 to 24': statistics.median([10,24]),
                     '25 to 49': statistics.median([25,49]),
                     '50 to 99': statistics.median([50,99]),
                     '100 to 149': statistics.median([100,149]),
                     '150 to 349': statistics.median([150,349]),
                     '350 or more': 350}

def hp_employee_ratio(x):

    # Company owners
    if x[2] == "Not asked":
        return round(hp_number_mapping[x[1]]/company_size_mapping[x[0]],1)
    # Sole traders
    elif x[1] == "Not asked":
        return round(hp_number_mapping[x[2]]/company_size_mapping[x[0]],1)
    else:
        raise ValueError
    
companysize_hpinstalls["HeatPumpPerEmployee"] = companysize_hpinstalls.apply(hp_employee_ratio, axis=1)
companysize_hpinstalls_crosstable = pd.crosstab(companysize_hpinstalls.CompanySize, companysize_hpinstalls.HeatPumpPerEmployee)
companysize_hpinstalls_crosstable.to_csv("../../outputs/other-statistics/hp_per_employee_crosstable.csv")
companysize_hpinstalls_crosstable

/tmp/ipykernel_3369/583758909.py:19: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if x[2] == "Not asked":
/tmp/ipykernel_3369/583758909.py:20: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  return round(hp_number_mapping[x[1]]/company_size_mapping[x[0]],1)
/tmp/ipykernel_3369/583758909.py:22: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  elif x[1] == "Not asked":
/tmp/ipykernel_3369/583758909.py:23: FutureWarning: Series.__getitem__

HeatPumpPerEmployee,0.0,0.1,0.3,0.6,0.7,1.1,1.2,1.7,2.0,2.4,2.5,4.0,4.8,5.0,5.7,8.0,12.3,17.0,24.8
CompanySize,,,,,,,,,,,,,,,,,,,
I’m a sole trader,3,0,0,0,0,0,0,0,0,0,0,0,0,33,0,0,0,2,0
I own a company with 5 or fewer employees,9,0,0,0,0,0,0,59,0,0,0,0,0,0,22,0,16,0,3
I own a company with 6-25 employees,2,0,5,0,0,10,0,0,0,10,0,0,8,0,0,4,0,0,0
I own a company with 26-100 employees,1,0,2,1,0,0,2,0,1,0,0,3,0,0,0,0,0,0,0
I own a company with over 100 employees,0,1,0,0,1,0,1,0,0,0,2,0,0,0,0,0,0,0,0
